In [3]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, GlobalAveragePooling2D, Dense, Multiply, Reshape, Conv2D, MaxPool2D, Flatten, Dropout, RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

In [4]:
# Define the custom AttentionBlock layer
class AttentionBlock(Layer):
    def __init__(self):
        super(AttentionBlock, self).__init__()

    def build(self, input_shape):
        self.channel_attention = Sequential([
            GlobalAveragePooling2D(),
            Dense(input_shape[-1] // 8, activation='relu'),
            Dense(input_shape[-1], activation='sigmoid'),
            Reshape((1, 1, input_shape[-1]))
        ])
        super(AttentionBlock, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.channel_attention(inputs)
        return Multiply()([inputs, attention_weights])

In [5]:
# Load the dataset
training_set = tf.keras.utils.image_dataset_from_directory(
    'train',
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=32,
    image_size=(128, 128),
    shuffle=True,
    seed=42,
    validation_split=None,
    subset=None,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

Found 70295 files belonging to 38 classes.


In [6]:
validation_set = tf.keras.utils.image_dataset_from_directory(
    'valid',
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=32,
    image_size=(128, 128),
    shuffle=True,
    seed=42,
    validation_split=None,
    subset=None,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

Found 17572 files belonging to 38 classes.


In [7]:
# Data Augmentation
data_augmentation = Sequential([
    RandomFlip("horizontal_and_vertical"),
    RandomRotation(0.2),
    RandomZoom(0.2),
])

In [8]:
# Load Pre-Trained Model
base_model = EfficientNetB0(include_top=False, input_shape=(128, 128, 3), weights='imagenet')
base_model.trainable = False  # Freeze the base model

In [15]:
# Build the Hybrid CNN with Attention
model = Sequential()


In [17]:
# Add data augmentation
model.add(data_augmentation)

In [19]:
# Add the pre-trained model
model.add(base_model)


In [21]:
# Add attention block
model.add(AttentionBlock())  # Add the custom attention layer

In [23]:
# Add custom layers
model.add(GlobalAveragePooling2D())
model.add(Dense(1500, activation='relu'))
model.add(Dropout(0.4))


In [25]:
# Output layer
model.add(Dense(38, activation='softmax'))  # 38 classes for leaf diseases


In [27]:
# Build the model by passing a batch of data through it
for images, _ in training_set.take(1):
    model(images)  # This builds the model

# Print model summary
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (32, 128, 128, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ efficientnetb0 (Functional)          │ (32, 4, 4, 1280)            │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ attention_block (AttentionBlock)     │ (32, 4, 4, 1280)            │         411,040 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (32, 1280)                  │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (32, 1500)                  │       1,921,500 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (32, 1500)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (32, 38)                    │          57,038 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,439,149 (24.56 MB)

 Trainable params: 2,389,578 (9.12 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [29]:
# Compile the Model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train the Model
initial_epochs = 10
print("\nStarting Training...")
training_history = model.fit(
    x=training_set,
    validation_data=validation_set,
    epochs=initial_epochs
)



Starting Training...
Epoch 1/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 2301s 1s/step - accuracy: 0.6258 - loss: 1.5006 - val_accuracy: 0.8635 - val_loss: 0.4576
Epoch 2/10
 401/2197 ━━━━━━━━━━━━━━━━━━━━ 24:28 818ms/step - accuracy: 0.8569 - loss: 0.4597